# 3 — Visualisation
All figures saved to `results/figures/`. Test set is never loaded here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, roc_curve, auc
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

FIG_DIR  = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('../data/preprocessed_data')
LOG_FILE = Path('../results/experiment_log.csv')

# load experiment log
log = pd.read_csv(LOG_FILE)
print(log[['experiment_id','model_type','cv_roc_auc_mean','cv_f1_mean','cv_recall_mean']].to_string(index=False))

# load train data
X_train = pd.read_csv(DATA_DIR / 'X_train.csv', index_col=0)
y_train = pd.read_csv(DATA_DIR / 'y_train.csv', index_col=0).squeeze()

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Chart 1 — Experiment Comparison (all metrics)

In [ ]:
metrics     = ['cv_accuracy_mean','cv_f1_mean','cv_roc_auc_mean','cv_precision_mean','cv_recall_mean']
labels      = ['Accuracy','F1','ROC-AUC','Precision','Recall']
exp_labels  = log['experiment_id'].tolist()
n_exp, n_met = len(exp_labels), len(metrics)
x    = np.arange(n_met)
w    = 0.15
colors = plt.cm.tab10(np.linspace(0, 0.5, n_exp))

fig, ax = plt.subplots(figsize=(12, 5))
for i, (exp, color) in enumerate(zip(exp_labels, colors)):
    row    = log[log['experiment_id'] == exp].iloc[0]
    values = [row[m] for m in metrics]
    std    = [row[m.replace('_mean','_std')] for m in metrics]
    bars   = ax.bar(x + i*w - (n_exp-1)*w/2, values, w,
                    label=exp, color=color, yerr=std,
                    capsize=3, error_kw={'linewidth':0.8})

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel('Score')
ax.set_title('Experiment Comparison — All Metrics (mean ± std, 5-fold CV)', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'chart1_experiment_comparison.png', bbox_inches='tight')
plt.show()

## Chart 2 — Metric Progression Across Experiments

In [ ]:
metric_map = {
    'cv_roc_auc_mean':   ('ROC-AUC',   'steelblue'),
    'cv_f1_mean':        ('F1',         'tomato'),
    'cv_recall_mean':    ('Recall',     'seagreen'),
    'cv_precision_mean': ('Precision',  'darkorange'),
    'cv_accuracy_mean':  ('Accuracy',   'mediumpurple'),
}

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(log))
for col, (label, color) in metric_map.items():
    ax.plot(x, log[col], marker='o', label=label, color=color, linewidth=2)
    ax.fill_between(
        x,
        log[col] - log[col.replace('_mean','_std')],
        log[col] + log[col.replace('_mean','_std')],
        alpha=0.1, color=color
    )

ax.set_xticks(list(x))
ax.set_xticklabels(log['experiment_id'], fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Metric Progression Across Experiments', fontsize=13)
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'chart2_metric_progression.png', bbox_inches='tight')
plt.show()

## Chart 3 — Feature Importance (exp_005 best params)

In [ ]:
best_model = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.01,
    min_child_weight=5, subsample=0.8, colsample_bytree=1.0,
    random_state=42, eval_metric='logloss', verbosity=0
)
best_model.fit(X_train, y_train)

importance = pd.Series(best_model.feature_importances_, index=X_train.columns)
importance = importance.sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tomato' if v == importance.max() else 'steelblue' for v in importance.values]
importance.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('Feature Importance — XGBoost Best Model (exp_005 params)', fontsize=13)
ax.axvline(importance.mean(), color='grey', linestyle='--', linewidth=1, label='mean')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'chart3_feature_importance.png', bbox_inches='tight')
plt.show()

## Chart 4 — CV Confusion Matrix (exp_005 best model + SMOTE)

In [ ]:
pipeline = ImbPipeline([
    ('sampler', SMOTE(random_state=42)),
    ('model',   XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.01,
        min_child_weight=5, subsample=0.8, colsample_bytree=1.0,
        random_state=42, eval_metric='logloss', verbosity=0
    ))
])

y_pred = cross_val_predict(pipeline, X_train, y_train, cv=CV)
cm     = confusion_matrix(y_train, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Negative','Pred Positive'],
    yticklabels=['True Negative','True Positive'],
    ax=ax, linewidths=0.5
)
ax.set_title('CV Confusion Matrix — exp_005 (train set, 5-fold)', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'chart4_confusion_matrix.png', bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  TN={tn}  FN={fn}')
print(f'Recall (sensitivity): {tp/(tp+fn):.3f}')
print(f'Specificity:          {tn/(tn+fp):.3f}')

## Chart 5 — ROC Curves (exp_002 vs exp_004 vs exp_005)

In [ ]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

def make_exp002():
    return XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        scale_pos_weight=1.87, random_state=42,
        eval_metric='logloss', verbosity=0
    )

def make_exp004():
    return ImbPipeline([
        ('sampler', SMOTE(random_state=42)),
        ('model', XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, eval_metric='logloss', verbosity=0
        ))
    ])

def make_exp005():
    return ImbPipeline([
        ('sampler', SMOTE(random_state=42)),
        ('model', XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.01,
            min_child_weight=5, subsample=0.8, colsample_bytree=1.0,
            random_state=42, eval_metric='logloss', verbosity=0
        ))
    ])

experiments = [
    ('exp_002 (XGB + scale_pos_weight)', make_exp002(), 'steelblue'),
    ('exp_004 (XGB + SMOTE)',            make_exp004(), 'tomato'),
    ('exp_005 (XGB + SMOTE + tuned)',    make_exp005(), 'seagreen'),
]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1], 'k--', linewidth=0.8, label='Random classifier')

for label, model, color in experiments:
    y_prob = cross_val_predict(model, X_train, y_train, cv=CV, method='predict_proba')[:,1]
    fpr, tpr, _ = roc_curve(y_train, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{label}  (AUC={roc_auc:.3f})', color=color, linewidth=2)

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Top 3 Models (5-fold CV, train set)', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'chart5_roc_curves.png', bbox_inches='tight')
plt.show()